# Comparación del barrido de learning rate

Lee `Resultados/sweep/*/` (`config.json`, `metrics.json`, `history.json`) y compara las corridas de una receta.
No entrena nada ni modifica archivos.

Receta por defecto: sin curriculum, batch 256, augmentation y scheduler activados (editar `RECETA` abajo).

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

SWEEP = Path("sweep") if Path("sweep").exists() else Path("Resultados/sweep")

runs = []
for d in sorted(SWEEP.iterdir()):
    m, c = d / "metrics.json", d / "config.json"
    if not (m.exists() and c.exists()):
        continue  # corrida incompleta
    cfg = json.loads(c.read_text(encoding="utf-8"))
    met = json.loads(m.read_text(encoding="utf-8"))
    runs.append(dict(
        run=d.name, arch=cfg["arch"], repr=cfg["repr"], params=met["parametros"], lr=cfg["lr"],
        batch=cfg["batch_size"], augment=cfg["augment"], curriculum=cfg["curriculum"],
        scheduler=cfg.get("lr_scheduler", "_sch" in d.name),
        acc=met["accuracy_global"], acc_low=met["accuracy_snr_menor_igual_-10"],
        acc_high=met["accuracy_snr_mayor_0"], epocas=met["epocas_entrenadas"],
        minutos=met["segundos_entrenamiento"] / 60, por_snr=met["accuracy_por_snr"], dir=d,
    ))
df = pd.DataFrame(runs)
print(f"{len(df)} corridas completas en {SWEEP}")

In [ ]:
# Receta a comparar: solo corridas que coincidan con estos valores
RECETA = dict(curriculum=False, batch=256, augment=True, scheduler=True)

sel = df[
    (df.curriculum == RECETA["curriculum"]) & (df.batch == RECETA["batch"]) &
    (df.augment == RECETA["augment"]) & (df.scheduler == RECETA["scheduler"])
].sort_values(["arch", "lr"]).reset_index(drop=True)

sel[["arch", "repr", "params", "lr", "acc", "acc_low", "acc_high", "epocas", "minutos"]]

In [ ]:
# Accuracy global y a SNR <= -10 dB por arquitectura y lr
etiquetas = sel.apply(lambda r: f"{r.arch} {r.repr}\nlr {r.lr:g}", axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, titulo in zip(axes, ["acc", "acc_low"], ["Accuracy global", "Accuracy a SNR <= -10 dB"]):
    ax.bar(etiquetas, sel[col], color=["#4C72B0" if lr == 1e-3 else "#DD8452" for lr in sel.lr])
    ax.set_title(titulo)
    ax.tick_params(axis="x", rotation=45)
    for i, v in enumerate(sel[col]):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de validación por época: se ve la inestabilidad y dónde corta el early stopping
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for _, r in sel.iterrows():
    h = json.loads((r.dir / "history.json").read_text(encoding="utf-8"))
    etiqueta = f"{r.arch} {r.repr} lr {r.lr:g}"
    axes[0].plot(h["val_acc"], label=etiqueta)
    axes[1].plot(h["val_loss"], label=etiqueta)
axes[0].set_title("Val accuracy por época")
axes[1].set_title("Val loss por época")
axes[1].set_yscale("log")
for ax in axes:
    ax.set_xlabel("Época")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Costo: minutos totales y minutos por época
sel_t = sel.assign(min_por_epoca=sel.minutos / sel.epocas)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(etiquetas, sel_t.minutos)
axes[0].set_title("Minutos totales")
axes[1].bar(etiquetas, sel_t.min_por_epoca)
axes[1].set_title("Minutos por época")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"Total: {sel.minutos.sum() / 60:.1f} horas")

In [ ]:
# Accuracy por SNR de cada corrida
fig, ax = plt.subplots(figsize=(10, 5))
for _, r in sel.iterrows():
    niveles = sorted(r.por_snr, key=int)
    ax.plot([int(s) for s in niveles], [r.por_snr[s] for s in niveles], marker="o",
            label=f"{r.arch} {r.repr} lr {r.lr:g}")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy por SNR")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()